# S02 · Find the unknowns — solving a small system

You know a couple of facts about some unknown numbers, and you want the computer to
work out the numbers. That is called **solving a system**, and it is one line of
code. We do it on a tiny canteen-pricing puzzle you can check in your head.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press play on each cell, top
  to bottom, and read the plain-English note above each one.
- New to the idea of a table of numbers? Open `primers/vectors_and_matrices.md`
  for a ten-minute, picture-first version.
- The main path is Steps 1 to 3. The **Stretch (optional)** cells at the end are a
  bonus for anyone who wants the deeper linear algebra (eigenvectors and the SVD).
  Skipping them costs you nothing.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, which Google Colab already ships.
# So there is nothing to install here.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np              # fast maths on lists and tables of numbers

np.random.seed(0)
print("NumPy is ready.")

## Step 1 — write down the facts

A canteen sells samosas and chai. You do not know the price of either, but you know
two facts from two receipts:

- 2 samosas and 1 chai came to ₹50.
- 1 samosa and 3 chai also came to ₹50.

Each fact is really "some number of samosas, some number of chai, equals a total".
We put the counts into a table `A` (one row per receipt) and the totals into a list
`b`. The unknown prices are what we are solving for.

In [ ]:
# Each row is one receipt: [samosas bought, chai bought].
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])

# The two totals, in rupees.
b = np.array([50.0, 50.0])

print("counts table A:")
print(A)
print()
print("totals b:", b)

## Step 2 — ask the computer for the unknown prices

`np.linalg.solve` takes the counts table and the totals and hands back the prices
that fit both facts at once. One line. It is faster and more accurate than the
by-hand method of working out an inverse, so this is the way to do it in code.

In [ ]:
# Solve for the unknown prices. The answer is [samosa price, chai price].
prices = np.linalg.solve(A, b)

samosa_price = prices[0]
chai_price = prices[1]

print("samosa price : Rs", round(samosa_price, 2))
print("chai price   : Rs", round(chai_price, 2))

## Step 3 — check the answer

Always check. If the prices are right, then plugging them back into the counts
table should reproduce the two totals. We multiply the table by the prices with
`@` (matrix multiply) and compare with `b`.

In [ ]:
# Multiply the counts table by the prices we found.
# @ means matrix multiply: it works out each receipt's total.
totals_check = A @ prices

print("totals from our prices :", totals_check)
print("the real totals        :", b)

# np.allclose is True if two arrays match up to tiny rounding.
print()
print("Do they match?", np.allclose(totals_check, b))

That is the whole main path. You gave the computer two facts and it found the
two unknown prices, and you checked them. The same one line, `np.linalg.solve`,
scales up to hundreds of unknowns without any extra work from you.

Everything below is an optional bonus for the mathematically curious. It is not
needed for the lab, and you can stop here with a clear conscience.

### Stretch (optional) — eigenvectors: the directions a table only stretches

A table of numbers can be seen as an action: multiply a vector by it and the vector
usually gets turned to point somewhere new. But a few special directions do not
turn at all; they only get longer or shorter. Those directions are the
**eigenvectors**, and the stretch factor for each is its **eigenvalue** `λ`, so
`A v = λ v`. On paper you might solve `det(A − λI) = 0`; `np.linalg.eig` returns the
same answers instantly. These directions are the engine behind PCA, which you meet
in Session 11.

In [ ]:
# A simple table that stretches along the two axes by different amounts.
M = np.array([[2.0, 0.0],
              [0.0, 3.0]])

# eig returns the eigenvalues and the eigenvectors.
eigenvalues, eigenvectors = np.linalg.eig(M)

print("eigenvalues (the stretch factors):", eigenvalues)
print()
print("eigenvectors (one per column):")
print(eigenvectors)

### Stretch (optional) — confirm A v = λ v

Let us verify the promise for the first eigenvalue and its eigenvector. The left
side `A v` and the right side `λ v` should be the very same vector.

In [ ]:
# Take the first eigenvalue and the first eigenvector (the first column).
first_eigenvalue = eigenvalues[0]
first_eigenvector = eigenvectors[:, 0]

print("first eigenvalue  :", first_eigenvalue)
print("first eigenvector :", first_eigenvector)

# Left side: the table times the eigenvector.
left_side = M @ first_eigenvector

# Right side: the eigenvalue times the eigenvector.
right_side = first_eigenvalue * first_eigenvector

print()
print("A v      :", left_side)
print("lambda v :", right_side)
print("Same?", np.allclose(left_side, right_side))

### Stretch (optional) — the SVD: split any table into rotate, stretch, rotate

The **singular value decomposition** breaks any table into three pieces,
`A = U @ S @ V-transpose`. The middle piece is a list of **singular values** that
say how much each direction is stretched. Unlike eigenvalues, the SVD exists for
every table, even a rectangular one. Keeping only the largest singular values is
exactly how image and data compression work.

In [ ]:
# A small rectangular 2-by-3 table.
B = np.array([[3.0, 1.0, 1.0],
              [1.0, 3.0, 1.0]])

# svd returns U, the singular values, and V-transpose (called Vt).
U, singular_values, Vt = np.linalg.svd(B)

print("U shape         :", U.shape)
print("singular values :", singular_values.round(3))
print("Vt shape        :", Vt.shape)

### Stretch (optional) — rebuild the table from its pieces

If the SVD is right, multiplying the three pieces back together returns the original
table. We first place the singular values on the diagonal of a middle matrix the
right size.

In [ ]:
# B has 2 rows and 3 columns, so the middle matrix S must be 2-by-3.
S = np.zeros((2, 3))
S[0, 0] = singular_values[0]
S[1, 1] = singular_values[1]

print("middle matrix S (singular values on the diagonal):")
print(S.round(3))

# Multiply the three pieces back together.
reconstructed = U @ S @ Vt

print()
print("rebuilt table:")
print(reconstructed.round(3))
print()
print("original table B:")
print(B)
print()
print("Do they match?", np.allclose(reconstructed, B))

## What you just did

On the main path you turned two facts into a counts table and a list of totals, and
one call to `np.linalg.solve` handed you the unknown prices, which you then checked.
That "here are the knowns, find the unknowns" move is everywhere in industry, from
pricing to scheduling to fitting models.

If you tried the Stretch cells, you also met the two most useful facts in linear
algebra: eigenvectors (the directions a table only stretches) and the SVD (splitting
any table into rotate, stretch, rotate). Both come back later in the course.

Next notebook: `03_fit_a_line_to_data.ipynb`, where we draw the best straight line
through a cloud of points and use it to predict a new value.